# Full Pipeline

## Imports

In [ ]:
import os
import cv2
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
import torchvision.transforms.v2 as T
from collections import defaultdict

pl.seed_everything(23)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MEAN = torch.tensor([0.485, 0.456, 0.406])
STD = torch.tensor([0.229, 0.224, 0.225])

def denormalize(tensor):
    m = MEAN.view(3, 1, 1).to(tensor.device)
    s = STD.view(3, 1, 1).to(tensor.device)
    return torch.clamp(tensor * s + m, 0, 1)

## Models

In [ ]:
class FenceSegmentationLightning(pl.LightningModule):
    def __init__(self, learning_rate=3e-4):
        super().__init__()
        self.model = smp.Unet(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=1)

    def forward(self, x):
        return self.model(x)

class LightningModuleInpainting(pl.LightningModule):
    def __init__(self, learning_rate=3e-4):
        super().__init__()
        # Entrada de 4 canales: RGB (3) + Mask (1)
        self.model = smp.Unet(encoder_name='resnet50', encoder_weights='imagenet', in_channels=4, classes=3)
        
    def forward(self, x):
        return self.model(x)

## Dataset

In [ ]:
class FenceInferenceDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.image_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(".jpg")])

    def __len__(self): return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        base = os.path.splitext(img_name)[0]
        image = Image.open(os.path.join(self.images_dir, img_name)).convert("RGB")
        mask = Image.open(os.path.join(self.labels_dir, base + ".png")).convert("L")

        if self.transforms:
            image, mask = self.transforms(image, mask)
        
        return {"image": image, "mask": (mask > 0.5).float(), "filename": img_name}

val_transforms = T.Compose([
    T.Resize(256, antialias=True),
    T.CenterCrop(224),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=MEAN.tolist(), std=STD.tolist()),
])

val_dataset = FenceInferenceDataset(
    images_dir="dataset/Test Set/Test_Images",
    labels_dir="dataset/Test Set/Test_Labels",
    transforms=val_transforms
)

## Pipeline

In [ ]:
class FenceRemovalPipelineSimple:
    def __init__(self, seg_ckpt, inp_ckpt):
        self.seg_model = FenceSegmentationLightning.load_from_checkpoint(seg_ckpt).to(DEVICE).eval()
        self.inp_model = LightningModuleInpainting.load_from_checkpoint(inp_ckpt).to(DEVICE).eval()

    def process(self, image_tensor, dilation_k=5, patch_size=224, halo_size=32):
        """
        Processes the image using a patch-based approach for both Segmentation and Inpainting.
        image_tensor: [1, 3, H, W] normalized
        """
        _, _, H, W = image_tensor.shape
        final_output = image_tensor.clone()
        full_dilated_mask = torch.zeros((1, 1, H, W), device=DEVICE)
        full_raw_mask = torch.zeros((1, 1, H, W), device=DEVICE)
        
        # Calculate patch coordinates
        patch_coords = get_patches_with_halo(H, W, patch_size, halo_size)

        with torch.no_grad():
            # Step 1: Patch-based Segmentation
            for c in patch_coords:
                y1, x1, y2, x2 = c['crop']
                patch = image_tensor[:, :, y1:y2, x1:x2]
                
                # Predict Mask
                seg_logits = self.seg_model(patch)
                patch_raw_mask = (torch.sigmoid(seg_logits) > 0.5).float()
                
                # Assign valid part to full mask
                vs_y1, vs_x1, vs_y2, vs_x2 = c['valid_src']
                vd_y1, vd_x1, vd_y2, vd_x2 = c['valid_dst']
                full_raw_mask[:, :, vd_y1:vd_y2, vd_x1:vd_x2] = patch_raw_mask[:, :, vs_y1:vs_y2, vs_x1:vs_x2]

            # Step 2: Dilation (safe)
            mask_np = full_raw_mask.squeeze().cpu().numpy()

            if dilation_k > 0:
                kernel = cv2.getStructuringElement(
                    cv2.MORPH_ELLIPSE, (dilation_k, dilation_k)
                )
                dilated_np = cv2.dilate(mask_np, kernel, iterations=1)
            else:
                dilated_np = mask_np

            full_dilated_mask = (
                torch.from_numpy(dilated_np)
                .unsqueeze(0)
                .unsqueeze(0)
                .to(DEVICE)
            )


            # Step 3: Patch-based Inpainting
            for c in patch_coords:
                y1, x1, y2, x2 = c['crop']
                img_patch = image_tensor[:, :, y1:y2, x1:x2]
                mask_patch = full_dilated_mask[:, :, y1:y2, x1:x2]
                
                # Prepare Inpainting input
                masked_patch = img_patch * (1 - mask_patch)
                inp_input = torch.cat([masked_patch, mask_patch], dim=1)
                
                # Predict Inpainting
                patch_pred = self.inp_model(inp_input)
                
                # Reconstruction: Only replace the pixels where the mask is 1
                patch_res = img_patch * (1 - mask_patch) + patch_pred * mask_patch
                
                # Assign valid part to full image
                vs_y1, vs_x1, vs_y2, vs_x2 = c['valid_src']
                vd_y1, vd_x1, vd_y2, vd_x2 = c['valid_dst']
                final_output[:, :, vd_y1:vd_y2, vd_x1:vd_x2] = patch_res[:, :, vs_y1:vs_y2, vs_x1:vs_x2]
            
        return final_output, full_dilated_mask, full_raw_mask

In [ ]:
class FenceRemovalPipelineHalo:
    def __init__(self, seg_ckpt, inp_ckpt):
        self.seg_model = FenceSegmentationLightning.load_from_checkpoint(seg_ckpt).to(DEVICE).eval()
        self.inp_model = LightningModuleInpainting.load_from_checkpoint(inp_ckpt).to(DEVICE).eval()

    def process(self, image_tensor, dilation_k=5, patch_size=224, halo_size=224):
        """
        Processes the image using a patch-based approach with context halo.
        
        Args:
            image_tensor: [1, 3, H, W] normalized
            patch_size: Size of the core patch to predict (e.g., 224)
            halo_size: Extra context around each patch (e.g., 224)
            
        The total extracted region will be (patch_size + 2*halo_size) x (patch_size + 2*halo_size),
        which gets resized down to 224x224 for the model, then resized back up.
        """
        _, _, H, W = image_tensor.shape
        
        # Adapt halo size to image dimensions (can't pad more than the dimension)
        # Leave at least 1 pixel for padding to work
        max_halo_h = max(1, H - 1)
        max_halo_w = max(1, W - 1)
        effective_halo = min(halo_size, max_halo_h, max_halo_w)
        
        # If image is smaller than patch_size, adjust patch_size too
        effective_patch = min(patch_size, H, W)
        stride = effective_patch  # Non-overlapping core patches
        
        # Pad image to handle edges
        img_padded = F.pad(image_tensor, (effective_halo, effective_halo, effective_halo, effective_halo), mode='reflect')
        
        full_raw_mask = torch.zeros((1, 1, H, W), device=DEVICE)
        count_mask = torch.zeros((1, 1, H, W), device=DEVICE)

        with torch.no_grad():
            # Step 1: Patch-based Segmentation with Context Halo
            for i in range(0, H, stride):
                for j in range(0, W, stride):
                    # Ensure we don't go out of bounds
                    i_safe = min(i, H - effective_patch)
                    j_safe = min(j, W - effective_patch)
                    
                    # Extract large context patch from padded image
                    # In padded coords, original (0,0) is at (effective_halo, effective_halo)
                    y_start = i_safe
                    x_start = j_safe
                    y_end = i_safe + effective_patch + (2 * effective_halo)
                    x_end = j_safe + effective_patch + (2 * effective_halo)
                    
                    large_patch = img_padded[:, :, y_start:y_end, x_start:x_end]
                    
                    # Resize to model input size (224x224)
                    input_patch = T.Resize((224, 224), antialias=True)(large_patch)
                    
                    # Predict mask
                    seg_logits = self.seg_model(input_patch)
                    probs = torch.sigmoid(seg_logits)
                    
                    # Resize mask back to context size
                    context_size = effective_patch + 2 * effective_halo
                    mask_large = T.Resize((context_size, context_size), antialias=True)(probs)
                    
                    # Crop out center (removing halo)
                    center_mask = mask_large[:, :, effective_halo:effective_halo+effective_patch, 
                                            effective_halo:effective_halo+effective_patch]
                    
                    # Accumulate
                    full_raw_mask[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += center_mask
                    count_mask[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += 1

            # Average overlapping predictions
            full_raw_mask = full_raw_mask / count_mask.clamp(min=1)
            binary_mask = (full_raw_mask > 0.5).float()

            # Step 2: Dilation (safe)
            mask_np = binary_mask.squeeze().cpu().numpy()

            if dilation_k > 0:
                kernel = cv2.getStructuringElement(
                    cv2.MORPH_ELLIPSE, (dilation_k, dilation_k)
                )
                dilated_np = cv2.dilate(mask_np, kernel, iterations=1)
            else:
                dilated_np = mask_np

            full_dilated_mask = (
                torch.from_numpy(dilated_np)
                .unsqueeze(0)
                .unsqueeze(0)
                .to(DEVICE)
            )

            # Step 3: Patch-based Inpainting with Context Halo
            final_output = torch.zeros_like(image_tensor)
            output_count = torch.zeros_like(image_tensor)
            
            # Pad dilated mask too
            mask_padded = F.pad(full_dilated_mask, (effective_halo, effective_halo, effective_halo, effective_halo), mode='reflect')

            for i in range(0, H, stride):
                for j in range(0, W, stride):
                    i_safe = min(i, H - effective_patch)
                    j_safe = min(j, W - effective_patch)
                    
                    # Extract large context patches
                    y_start = i_safe
                    x_start = j_safe
                    y_end = i_safe + effective_patch + (2 * effective_halo)
                    x_end = j_safe + effective_patch + (2 * effective_halo)
                    
                    img_large = img_padded[:, :, y_start:y_end, x_start:x_end]
                    mask_large = mask_padded[:, :, y_start:y_end, x_start:x_end]
                    
                    # Resize to model size
                    img_resized = T.Resize((224, 224), antialias=True)(img_large)
                    mask_resized = T.Resize((224, 224), antialias=True)(mask_large)
                    
                    # Prepare inpainting input
                    masked_img = img_resized * (1 - mask_resized)
                    inp_input = torch.cat([masked_img, mask_resized], dim=1)
                    
                    # Predict
                    pred_resized = self.inp_model(inp_input)
                    
                    # Resize back to context size
                    context_size = effective_patch + 2 * effective_halo
                    pred_large = T.Resize((context_size, context_size), antialias=True)(pred_resized)
                    img_large_result = img_large * (1 - mask_large) + pred_large * mask_large
                    
                    # Crop center
                    center_result = img_large_result[:, :, effective_halo:effective_halo+effective_patch, 
                                                     effective_halo:effective_halo+effective_patch]
                    
                    # Accumulate
                    final_output[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += center_result
                    output_count[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += 1
            
            final_output = final_output / output_count.clamp(min=1)
            
        return final_output, full_dilated_mask, binary_mask

In [ ]:
# Cell: Imports (Update to include these utilities)
def get_patches_with_halo(image_height, image_width, patch_size, halo_size):
    coords = []
    stride = patch_size - 2 * halo_size
    for y in range(0, image_height, stride):
        for x in range(0, image_width, stride):
            # Calculate patch bounds (with halo)
            y1 = max(0, y - halo_size)
            x1 = max(0, x - halo_size)
            y2 = min(image_height, y + patch_size - halo_size)
            x2 = min(image_width, x + patch_size - halo_size)
            
            # Adjust if patch hits the right/bottom edge
            if y2 == image_height: 
                y1 = max(0, y2 - patch_size)
            if x2 == image_width:  
                x1 = max(0, x2 - patch_size)
            
            # Valid region to extract from the resulting patch
            v_y1 = y - y1
            v_x1 = x - x1
            v_y2 = v_y1 + stride if y + stride <= image_height else v_y1 + (image_height - y)
            v_x2 = v_x1 + stride if x + stride <= image_width else v_x1 + (image_width - x)
            
            coords.append({
                'crop': (y1, x1, y1 + patch_size, x1 + patch_size),
                'valid_src': (v_y1, v_x1, v_y2, v_x2),
                'valid_dst': (y, x, y + (v_y2 - v_y1), x + (v_x2 - v_x1))
            })
    return coords

## Visualize Results

In [ ]:
def visualize_paper_results(pipeline, dataset, num_images=5):
    fig, axes = plt.subplots(num_images, 4, figsize=(20, 5 * num_images))
    
    for i in range(num_images):
        sample = dataset[i]
        img_t = sample["image"].unsqueeze(0).to(DEVICE)
        
        final_t, dilated_mask, raw_mask = pipeline.process(img_t, dilation_k=5)
        
        img_orig = denormalize(img_t[0]).cpu().permute(1, 2, 0).numpy()
        mask_det = raw_mask[0, 0].cpu().numpy()
        mask_dil = dilated_mask[0, 0].cpu().numpy()
        img_final = denormalize(final_t[0]).cpu().permute(1, 2, 0).numpy()
        
        axes[i, 0].imshow(img_orig)
        axes[i, 0].set_title("Input Photo", fontweight='bold')
        
        axes[i, 1].imshow(mask_det, cmap='magma')
        axes[i, 1].set_title("Neural Detection", fontweight='bold')
        
        axes[i, 2].imshow(mask_dil, cmap='gray')
        axes[i, 2].set_title("Refined Mask (Dilation)", fontweight='bold')
        
        axes[i, 3].imshow(img_final)
        axes[i, 3].set_title("Final De-fenced Result", fontweight='bold')
        
        for ax in axes[i]: ax.axis('off')

    plt.tight_layout()
    plt.savefig("paper_results_comparison.png", dpi=300)
    plt.show()

In [ ]:
import gradio as gr
import torch
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms.v2 as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- LOAD PIPELINE ---
pipeline_simple = FenceRemovalPipelineSimple(
    seg_ckpt="checkpoints/defence_model.ckpt",
    inp_ckpt="checkpoints/augmentation_mask_mix.ckpt"
)

pipeline_complex = FenceRemovalPipelineHalo(
    seg_ckpt="checkpoints/defence_model.ckpt",
    inp_ckpt="checkpoints/augmentation_mask_mix.ckpt"
)


# --- GRADIO FUNCTIONS ---
# Update these functions in the Gradio section
def auto_detect_simple(input_dict):
    img_pil = input_dict["background"].convert("RGB")

    transform = T.Compose([
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=MEAN.tolist(), std=STD.tolist())
    ])
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

    _, dilated_mask, _ = pipeline_simple.process(
        img_tensor,
        dilation_k=7,
        patch_size=224,
        halo_size=32
    )

    return build_rgba_mask(img_pil, dilated_mask)

def auto_detect_complex(input_dict):
    img_pil = input_dict["background"].convert("RGB")

    transform = T.Compose([
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=MEAN.tolist(), std=STD.tolist())
    ])
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

    _, dilated_mask, _ = pipeline_complex.process(
        img_tensor,
        dilation_k=7,
        patch_size=224,
        halo_size=224
    )

    return build_rgba_mask(img_pil, dilated_mask)

def build_rgba_mask(img_pil, dilated_mask):
    H, W = img_pil.height, img_pil.width

    mask_np = (dilated_mask.squeeze().cpu().numpy() * 255).astype(np.uint8)

    # Force exact spatial alignment
    mask_np = cv2.resize(
        mask_np,
        (W, H),
        interpolation=cv2.INTER_NEAREST
    )

    rgba_mask = np.zeros((H, W, 4), dtype=np.uint8)
    rgba_mask[..., :3] = 255
    rgba_mask[..., 3] = mask_np

    mask_pil = Image.fromarray(rgba_mask, mode="RGBA")

    return {
        "background": img_pil,
        "layers": [mask_pil],
        "composite": None
    }


def delete_fence(input_dict, mode):
    img_pil = input_dict["background"].convert("RGB")
    mask_pil = input_dict["layers"][0].convert("L")

    transform = T.Compose([
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=MEAN.tolist(), std=STD.tolist())
    ])

    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

    # Manual mask from Gradio (already refined)
    mask_tensor = (
        T.Compose([
            T.ToImage(),
            T.ToDtype(torch.float32, scale=True)
        ])(mask_pil) > 0.5
    ).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        if mode == "complex":
            # Context-aware inpainting
            final_output, _, _ = pipeline_complex.process(
                img_tensor,
                dilation_k=5,  # IMPORTANT: no dilation
                patch_size=224,
                halo_size=224
            )
        else:
            # Fast local inpainting
            final_output, _, _ = pipeline_simple.process(
                img_tensor,
                dilation_k=5,  # IMPORTANT: no dilation
                patch_size=224,
                halo_size=32
            )

    res_pil = T.ToPILImage()(
        denormalize(final_output.squeeze(0)).cpu()
    )
    return res_pil




# --- GRADIO UI ---
with gr.Blocks() as demo:
    gr.Markdown("# 🪚 DeFence Inpainter (Auto + Manual)")
    gr.Markdown("1️⃣ Upload image 2️⃣ Click 'Auto Detect Mask' 3️⃣ Refine mask with brush 4️⃣ Click 'Delete Fence'")
    
    with gr.Row():
        input_editor = gr.ImageEditor(
            label="Image + Mask Editor",
            type="pil",
            sources=["upload"],
            brush=gr.Brush(colors=["#FFFFFF"], default_size=20),
            layers=True
        )
        output_img = gr.Image(label="Result", type="pil")

    gr.Markdown("""
    ### Fence Detection Mode
    - **Simple Detection**: Fast, best for thin or regular fences
    - **Complex Detection**: Slower, better for thick or large fence structures
    """)

    mode_state = gr.State("simple")  # default
    with gr.Row():
        detect_simple_btn = gr.Button("Simple Fence Detection")
        detect_complex_btn = gr.Button("Complex Fence Detection")
        delete_btn = gr.Button("Delete Fence", variant="primary")
        
    
    detect_simple_btn.click(
        fn=lambda x: (auto_detect_simple(x), "simple"),
        inputs=input_editor,
        outputs=[input_editor, mode_state]
    )


    detect_complex_btn.click(
        fn=lambda x: (auto_detect_complex(x), "complex"),
        inputs=input_editor,
        outputs=[input_editor, mode_state]
    )

    delete_btn.click(
        fn=delete_fence,
        inputs=[input_editor, mode_state],
        outputs=output_img
    )



demo.launch(inline=True)
